# Recommendation System

In [1]:
!!pip install scikit-surprise


['Collecting scikit-surprise',
 '  Downloading scikit_surprise-1.1.4.tar.gz (154 kB)',
 '  Installing build dependencies: started',
 "  Installing build dependencies: finished with status 'done'",
 '  Getting requirements to build wheel: started',
 "  Getting requirements to build wheel: finished with status 'error'",
 '  error: subprocess-exited-with-error',
 '  ',
 '  × Getting requirements to build wheel did not run successfully.',
 '  │ exit code: 1',
 '  ╰─> [44 lines of output]',
 '      Compiling surprise/similarities.pyx because it changed.',
 '      Compiling surprise/prediction_algorithms/matrix_factorization.pyx because it changed.',
 '      Compiling surprise/prediction_algorithms/optimize_baselines.pyx because it changed.',
 '      Compiling surprise/prediction_algorithms/slope_one.pyx because it changed.',
 '      Compiling surprise/prediction_algorithms/co_clustering.pyx because it changed.',
 '      [1/5] Cythonizing surprise/prediction_algorithms/co_clustering.pyx',
 '

### Importing the Libaries

In [22]:
import pandas as pd
import numpy as np

from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity


In [23]:
df = pd.read_csv("../data/processed/recommender_dataset.csv")

print(df.head())
print("Shape:", df.shape)


   UserId  AttractionId  Rating
0   70456           640       5
1    7567           640       5
2   79069           640       5
3   31019           640       3
4   43611           640       3
Shape: (52930, 3)


In [24]:
df.dropna(inplace=True)

In [25]:
user_item_matrix = df.pivot_table(
    index="UserId",
    columns="AttractionId",
    values="Rating"
).fillna(0)

print("User-Item matrix shape:", user_item_matrix.shape)


User-Item matrix shape: (33530, 30)


In [26]:
n_items = user_item_matrix.shape[1]
n_components = min(15, n_items - 1)   # safe automatic choice


In [12]:
n_items = user_item_matrix.shape[1]
n_components = min(15, n_items - 1)   # safe automatic choice


In [27]:
svd = TruncatedSVD(n_components=n_components, random_state=42)

latent_matrix = svd.fit_transform(user_item_matrix)


In [28]:
reconstructed = np.dot(latent_matrix, svd.components_)

predicted_ratings = pd.DataFrame(
    reconstructed,
    index=user_item_matrix.index,
    columns=user_item_matrix.columns
)


In [ ]:
user_item_matrix[mask].dropna(inplace=True)
predicted_ratings.dropna(inplace=True)

In [31]:
# Get indices of known ratings
known_users, known_items = np.where(user_item_matrix.values > 0)

# Extract true and predicted ratings only at known positions
true_ratings = user_item_matrix.values[known_users, known_items]
pred_ratings = predicted_ratings.values[known_users, known_items]

# Compute RMSE safely
rmse = np.sqrt(mean_squared_error(true_ratings, pred_ratings))

print("Collaborative Filtering RMSE:", round(rmse, 4))


Collaborative Filtering RMSE: 0.7512


In [32]:
def recommend_collaborative(user_id, n=5):
    
    # Predicted ratings for this user
    user_preds = predicted_ratings.loc[user_id]
    
    # Items already rated
    rated_items = user_item_matrix.loc[user_id]
    
    # Keep only unseen items
    unseen_preds = user_preds[rated_items == 0]
    
    # Top-N highest predicted ratings
    top_n = unseen_preds.sort_values(ascending=False).head(n)
    
    return top_n


In [33]:
sample_user = user_item_matrix.index[0]

recommend_collaborative(sample_user, n=5)


AttractionId
1238    0.008231
1220    0.005033
1280    0.004087
897     0.003158
913     0.002160
Name: 14, dtype: float64

In [34]:
items = pd.read_excel("../data/raw/Item.xlsx")
types = pd.read_excel("../data/raw/Type.xlsx")

items = items.merge(types, on="AttractionTypeId", how="left")


In [35]:
content_features = pd.get_dummies(
    items[["AttractionId", "AttractionTypeId"]],
    columns=["AttractionTypeId"]
).set_index("AttractionId")


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(content_features)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=content_features.index,
    columns=content_features.index
)


In [37]:
def recommend_content(attraction_id, n=5):
    sims = similarity_df.loc[attraction_id].drop(attraction_id)
    return sims.sort_values(ascending=False).head(n)


In [38]:
def recommend_hybrid(user_id, n=5, alpha=0.7):
    
    collab_scores = predicted_ratings.loc[user_id]
    rated_items = user_item_matrix.loc[user_id]
    unseen = collab_scores[rated_items == 0]
    
    hybrid_scores = {}
    
    for item in unseen.index:
        content_score = similarity_df[item].mean()
        hybrid_scores[item] = alpha * unseen[item] + (1 - alpha) * content_score
    
    hybrid_series = pd.Series(hybrid_scores)
    
    return hybrid_series.sort_values(ascending=False).head(n)


In [39]:
def recommend_hybrid(user_id, n=5, alpha=0.7):
    
    collab_scores = predicted_ratings.loc[user_id]
    rated_items = user_item_matrix.loc[user_id]
    unseen = collab_scores[rated_items == 0]
    
    hybrid_scores = {}
    
    for item in unseen.index:
        content_score = similarity_df[item].mean()
        hybrid_scores[item] = alpha * unseen[item] + (1 - alpha) * content_score
    
    hybrid_series = pd.Series(hybrid_scores)
    
    return hybrid_series.sort_values(ascending=False).head(n)


In [40]:
predicted_ratings.to_csv("models/predicted_ratings.csv")


In [41]:
print(type(predicted_ratings.index[0]))
print(predicted_ratings.index[:5])


<class 'numpy.int64'>
Index([14, 16, 20, 23, 25], dtype='int64', name='UserId')
